**STUDYING THE ERRORS**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("train.csv")

no_feature_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
                   "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                   "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                   "MasVnrType"]
df[no_feature_cols] = df[no_feature_cols].fillna("None")
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

train,test = train_test_split(df, test_size=0.2, random_state=42)
train = train.copy()
test = test.copy()
median_frontage = train["LotFrontage"].median()
train["LotFrontage"] = train["LotFrontage"].fillna(median_frontage)
test["LotFrontage"] = test["LotFrontage"].fillna(median_frontage)

y_train = train["SalePrice"]
y_test = test["SalePrice"]
X_train = pd.get_dummies(train.drop(columns=["SalePrice", "Id"]), dtype=int)
X_test = pd.get_dummies(test.drop(columns=["SalePrice", "Id"]), dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

res = test[["Neighborhood", "OverallQual", "GrLivArea"]].copy()
res["actual"] = y_test
res["predicted"] = gb.predict(X_test).round()
res["error"] = res["predicted"] - res["actual"]
res["abs_error"] = res["error"].abs()
res["pct_error"] = (res["abs_error"] / res["actual"] * 100).round(1)

print("MAE:", round(res["abs_error"].mean()))
print("RMSE:", round(np.sqrt((res["error"] ** 2).mean())))
print("Median miss:", round(res["abs_error"].median()))
print("Median percent error:", round(res["pct_error"].median(), 1), "%")
print("Houses within 10% of the real price:", round((res["pct_error"] <= 10).mean() * 100), "%")
print("Houses within 20% of the real price:", round((res["pct_error"] <= 20).mean() * 100), "%")

MAE: 16626
RMSE: 27051
Median miss: 10453
Median percent error: 6.6 %
Houses within 10% of the real price: 69 %
Houses within 20% of the real price: 90 %


*The model’s median miss is 10,453 and 69% of houses are within 10% of the real price (90% within 20%). RMSE (27,051) is about 63% higher than MAE (16,626), which means a small number of houses are predicted badly and pull the error up.*

**FIND WHERE THE MODEL FAILS**

In [ ]:
worst = res.sort_values("abs_error", ascending=False).head(10)

print("Average signed error (negative = guesses too low):", round(res["error"].mean()))
print("Share of all squared error from the worst 10 houses:",
      round((worst["error"] ** 2).sum() / (res["error"] ** 2).sum() * 100), "%")
print()
print(worst[["Neighborhood", "OverallQual", "GrLivArea", "actual", "predicted", "error"]])
print()

res["price_band"] = pd.cut(res["actual"], [0, 150000, 250000, 400000, 1000000],
                           labels=["under 150K", "150K-250K", "250K-400K", "over 400K"])
print(res.groupby("price_band", observed=True).agg(
    houses=("actual", "count"),
    MAE=("abs_error", "mean"),
    avg_signed_error = ("error", "mean"),
    median_pct=("pct_error", "median")).round(1))

Average signed error (negative = guesses too low): -748
Share of all squared error from the worst 10 houses: 57 %

     Neighborhood  OverallQual  GrLivArea  actual  predicted     error
898       NridgHt            9       2364  611657   399104.0 -212553.0
581       NridgHt            8       2042  253293   374078.0  120785.0
1046      StoneBr            9       2868  556581   444963.0 -111618.0
774       NridgHt            8       1973  395000   285037.0 -109963.0
691       NoRidge           10       4316  755000   645308.0 -109692.0
261       CollgCr            8       2574  276000   360773.0   84773.0
218       Crawfor            7       1954  311500   236487.0  -75013.0
588       ClearCr            5       1473  143000   213652.0   70652.0
1024       Timber            8       2898  287000   356602.0   69602.0
1243      NridgHt           10       2076  465000   404418.0  -60582.0

            houses      MAE  avg_signed_error  median_pct
price_band                                   

*The model’s errors balance out overall (average signed error -748), but the over-400,000 band is underpredicted on all seven houses (average signed error -81,903, equal to its MAE), so the model tends to pull the most expensive houses toward the middle. Grouping by actual price exaggerates this effect, and with only seven houses the result is rough.*

*The worst 10 of 292 houses cause 57% of the squared error. Eight of them have quality 8 or higher, six were underpredicted and four overpredicted.*

In [ ]:
res["SaleCondition"] = test["SaleCondition"]
worst = res.sort_values("abs_error", ascending=False).head(10)

print(res.loc[[581, 261, 588, 1024], ["Neighborhood", "SaleCondition", "actual", "predicted"]])
print()
print("Worst 10:")
print(worst["SaleCondition"].value_counts())
print()
print("All test houses:")
print(res["SaleCondition"].value_counts())

     Neighborhood SaleCondition  actual  predicted
581       NridgHt       Partial  253293   374078.0
261       CollgCr       Partial  276000   360773.0
588       ClearCr       Partial  143000   213652.0
1024       Timber       Abnorml  287000   356602.0

Worst 10:
SaleCondition
Partial    7
Normal     2
Abnorml    1
Name: count, dtype: int64

All test houses:
SaleCondition
Normal     234
Partial     27
Abnorml     24
Alloca       5
Family       2
Name: count, dtype: int64


*All four overpredicted houses in the worst 10 were unusual sales (three Partial, one Abnorml). Seven of the 10 worst misses are Partial sales, although Partial sales are only 9% of the test set. About 26% of Partial houses are in the worst 10, against under 1% of Normal sales. Partial sales are mispredicted in both directions, so they are hard to price rather than simply overvalued. This is a strong clue from a small sample, not proof.*

In [5]:
cond = res["SaleCondition"]
print("MAE, Normal sales:", round(res.loc[cond == "Normal", "abs_error"].mean()))
print("MAE, Partial sales:", round(res.loc[cond == "Partial", "abs_error"].mean()))
print("MAE, Abnorml sales:", round(res.loc[cond == "Abnorml", "abs_error"].mean()))
print("Share of all absolute error from Partial and Abnorml sales:",
      round(res.loc[cond.isin(["Partial", "Abnorml"]), "abs_error"].sum() / res["abs_error"].sum() * 100), "%")

MAE, Normal sales: 13379
MAE, Partial sales: 42414
MAE, Abnorml sales: 15813
Share of all absolute error from Partial and Abnorml sales: 31 %


*Average error depends on the sale type: 13,379 for Normal sales, 15,813 for Abnormal, and 42,414 for Partial (unfinished new construction), over three times higher. Partial and Abnormal sales are 17% of the test houses but cause 31% of the absolute error. Part of the Partial gap may come from their higher prices, since dollar errors grow with price.*